# Prometheus – LLM Judge

In [ ]:
# PROMETHEUS – LLM AS A JUDGE
# --------------------------------------------------
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

In [ ]:
import torch, datetime, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
print("Loading Prometheus (4‑bit) …")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Unbabel/M-Prometheus-7B",
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "Unbabel/M-Prometheus-7B",
    trust_remote_code=True,
)

model.eval()

In [ ]:
def judge(to_predict, prediction, gold, verbose=False):
    # ---------------- template with named placeholders ----------------
    template = '''###Task Description:
An instruction, a response to evaluate, a **gold translation** (reference answer) and a score rubric representing an evaluation criterion are given.

1. Write a detailed feedback that assesses the response strictly according to the score rubric (translation quality), **not evaluating in general**.
2. After writing the feedback, write a score that is an integer between 1 and 5. You must refer to the rubric.
3. The output format must be EXACTLY:
"Feedback: <your feedback here>
[RESULT] <integer 1-5>"
4. Do not generate any headers, closings, or extra commentary.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Translation Quality):
5 – The translation preserves *all* meaning nuances of the archaic text, reads naturally in contemporary Italian, has no grammatical or lexical errors.  
4 – Very accurate; minor nuance or style issues that do not alter meaning. Fluency is near-native.  
3 – Generally preserves main meaning, but contains several inaccuracies, awkward phrasing, or scattered grammar errors.  
2 – Significant meaning loss or distortions. Fluent Italian is spotty; multiple grammar or word-choice errors.  
1 – Fails to convey core meaning; pervasive errors make the text hard or impossible to understand.

###Feedback:'''
    prompt = template.format(
        ARCHAIC_PASSAGE=to_predict.strip(),
        MODEL_TRANSLATION=prediction.strip(),
        GOLD_TRANSLATION=gold.strip()
    )

    # 2️⃣  Run Prometheus 2 --------------------------
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    if verbose:
        print("▶ Generating judgement ...")
        t0 = datetime.datetime.now()

    with torch.no_grad():
        out = model.generate(**inputs, max_length=1024, do_sample=False)

    raw = tokenizer.decode(out[0], skip_special_tokens=True)

    if verbose:
        print("⏱", datetime.datetime.now() - t0)
        print(raw)

    # 3️⃣  Parse feedback + numeric score ------------
    m = re.search(r"Feedback:\s*(.*?)\s*\[RESULT\]\s*([1-5])", raw, re.S)
    if not m:
        raise ValueError("Cannot parse Prometheus output:\n" + raw)

    feedback = m.group(1).strip()
    score    = int(m.group(2))

    return {"feedback": feedback, "score": score, "raw": raw}


In [ ]:
scores = []
feedbacks = []
for row in translated_dataset:
    to_predict = row["Sentence"]
    gold = row["TRADUZIONE ITALIANA"]
    prediction = row["prediction"]
    judgement = judge(to_predict, gold, prediction)
    scores.append(judgement["score"])
    feedbacks.append(judgement["feedback"])